# 房价与租金预测模型 - Team10


## 1. 数据预处理

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
import lightgbm as lgb
from lightgbm import LGBMRegressor
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BertModel, BertTokenizer
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle

tqdm.pandas()

class UnifiedDataProcessor:
    """统一数据处理类 - 整合三位成员的预处理方法"""
    
    def __init__(self, data_type='price', target='Price'):
        self.data_type = data_type  # 'price' or 'rent'
        self.target = target
        self.train_data = None
        self.test_data = None
        
        # 加载字段信息
        if data_type == 'price':
            columns_json = './data/data_info/price_info.json'
        else:
            columns_json = './data/data_info/rent_info.json'
            
        with open(columns_json, 'r', encoding='utf-8') as f:
            self.columns_info = json.load(f)

        # 配置参数
        self.MISSING_THRESHOLD = 0.2
        self.DROP_COLS = ['开发商', '物业公司', '抵押信息', '板块comm', '别墅类型', '抵押信息']
        
        # 加载模型
        try:
            self.kmeans = joblib.load('./src/model/kmeans_model.pkl')
            self.model = AutoModelForSequenceClassification.from_pretrained("jackietung/bert-base-chinese-sentiment-finetuned")
            self.tokenizer = AutoTokenizer.from_pretrained("jackietung/bert-base-chinese-sentiment-finetuned")
        except:
            print("部分模型加载失败，将继续使用基础功能")
    
    def load_data(self, train_path, test_path):
        """加载数据"""
        self.train_data = pd.read_csv(train_path, low_memory=False)
        self.test_data = pd.read_csv(test_path, low_memory=False)
        return self.train_data, self.test_data
    
    def clean_column_names(self, df):
        """清洗列名"""
        df.columns = df.columns.str.strip().str.replace(r'[^\w\u4e00-\u9fa5]', '', regex=True)
        df = df.loc[:, ~df.columns.duplicated()]
        return df
    
    def extract_numeric_from_unit(self, df):
        """从带单位文本中提取数值"""
        unit_cols = ['燃气费', '供热费', '停车费用', '物业费', '房屋总数', '楼栋总数', '绿化率', '建筑面积', '面积']
        
        for col in unit_cols:
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: float(re.findall(r'[-+]?\d+\.?\d*', str(x))[0]) 
                    if re.findall(r'[-+]?\d+\.?\d*', str(x)) else np.nan
                )
        return df
    
    def process_construction_year(self, df):
        """处理建筑年代"""
        if '建筑年代' in df.columns:
            def process_year(year_str):
                if pd.isna(year_str):
                    return np.nan
                year_str = str(year_str).replace('年', '').strip()
                if '-' in year_str:
                    parts = year_str.split('-')
                    if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
                        return (float(parts[0]) + float(parts[1])) / 2
                return float(year_str) if year_str.isdigit() else np.nan
            
            df['建筑年代'] = df['建筑年代'].apply(process_year)
        return df
    
    def split_ladder_household(self, df):
        """拆分梯户比例"""
        if '梯户比例' in df.columns:
            def split_ratio(ratio_str):
                if pd.isna(ratio_str):
                    return pd.Series([0, 0])
                ratio_str = str(ratio_str).strip()
                chinese_map = {'零':0,'一':1,'二':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9,'十':10}
                
                ladder_part = re.split(r'梯', ratio_str)[0] if '梯' in ratio_str else ''
                ladder = chinese_map.get(ladder_part, 0)
                household = 0
                
                if '梯' in ratio_str:
                    household_part = re.split(r'梯', ratio_str)[1]
                    household_part = re.split(r'户', household_part)[0] if '户' in household_part else household_part
                    household = chinese_map.get(household_part, 0)
                return pd.Series([ladder, household])
            
            ladder_household = df['梯户比例'].apply(split_ratio)
            ladder_household.columns = ['梯num', '户num']
            df = pd.concat([df, ladder_household], axis=1)
            df = df.drop(columns=['梯户比例'])
        return df
    
    def extract_room_features(self, df):
        """提取房屋户型特征"""
        house_type_cols = [col for col in df.columns if any(kw in col for kw in ['户型', '房型', '房屋户型'])]
        
        if house_type_cols:
            house_type_col = house_type_cols[0]
            
            def extract_rooms(house_type):
                if pd.isna(house_type) or house_type == '':
                    return pd.Series([0, 0, 0, 0])
                
                house_type = str(house_type)
                if "房间" in house_type:
                    house_type = "1室1厅1厨1卫"
                
                room_info = re.findall(r'(\d+)室|(\d+)厅|(\d+)厨|(\d+)卫', house_type)
                room_dict = {'室': 0, '厅': 0, '厨': 0, '卫': 0}
                
                for match in room_info:
                    if match[0]: room_dict['室'] = int(match[0])
                    if match[1]: room_dict['厅'] = int(match[1])
                    if match[2]: room_dict['厨'] = int(match[2])
                    if match[3]: room_dict['卫'] = int(match[3])
                
                return pd.Series([room_dict['室'], room_dict['厅'], room_dict['厨'], room_dict['卫']])
            
            room_features = df[house_type_col].apply(extract_rooms)
            room_features.columns = ['室num', '厅num', '厨num', '卫num']
            df = pd.concat([df, room_features], axis=1)
            df = df.drop(columns=[house_type_col])
        else:
            df['室num'], df['厅num'], df['厨num'], df['卫num'] = 0, 0, 0, 0
            
        return df
    
    def calculate_orientation_score(self, orientation):
        """计算房屋朝向得分"""
        if pd.isna(orientation) or orientation == '':
            return 0
        
        score = 0
        orientation = str(orientation)
        
        single_directions = ['东', '南', '西', '北']
        for dir in single_directions:
            if dir in orientation:
                score += 1

        double_directions = ['东南', '西南', '东北', '西北', '东南西', '南北', '东南北', '南北东']
        for dir in double_directions:
            if dir in orientation:
                score += 0.5

        if '南 北' in orientation:
            score += 2

        return score
    
    def process_text_features(self, df):
        """处理文本特征"""
        text_cols = ['客户反馈', '房屋优势', '核心卖点', '周边配套', '交通出行']
        text_cols = [c for c in text_cols if c in df.columns]
        
        if text_cols:
            df['text_combined_raw'] = df[text_cols].apply(lambda x: " ".join(x.astype(str)), axis=1)
            
            for col in text_cols:
                for kw in ['地铁', '学区', '公园', '医院', '满五年', '精装', '南北通透']:
                    df[f'{col}_has_{kw}'] = df[col].apply(lambda x: 1 if kw in str(x) else 0)
            
            df = df.drop(columns=text_cols, errors='ignore')
        
        return df
    
    def add_advanced_features(self, df):
        """添加高级特征"""
        # 朝向得分
        if '房屋朝向' in df.columns:
            df['朝向得分'] = df['房屋朝向'].apply(self.calculate_orientation_score)
        
        # 聚类特征
        if all(col in df.columns for col in ['lon', 'lat']):
            try:
                df['cluster_label'] = self.kmeans.predict(df[['lon', 'lat']])
            except:
                df['cluster_label'] = 0
        
        # 交易时间特征
        time_cols = ['交易时间', '上次交易']
        if all(col in df.columns for col in time_cols):
            df['交易时间_dt'] = pd.to_datetime(df['交易时间'], errors='coerce')
            df['上次交易_dt'] = pd.to_datetime(df['上次交易'], errors='coerce')
            df['交易间隔_天'] = (df['交易时间_dt'] - df['上次交易_dt']).dt.days
            df = df.drop(columns=['交易时间_dt', '上次交易_dt'])
        
        # 户均楼栋房屋数
        if all(col in df.columns for col in ['房屋总数', '楼栋总数']):
            df['户均楼栋房屋数'] = df['房屋总数'] / df['楼栋总数']
        
        # 对数变换
        area_col = '建筑面积' if '建筑面积' in df.columns else '面积'
        if area_col in df.columns:
            df['log_' + area_col] = np.log(df[area_col].replace(0, 0.1))
        
        if self.target in df.columns:
            df['log_' + self.target] = np.log(df[self.target])
        
        return df
    
    def fill_missing_values(self):
        """填充缺失值"""
        for df in [self.train_data, self.test_data]:
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            categorical_cols = df.select_dtypes(exclude=[np.number]).columns
            
            # 数值列用中位数填充
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
            
            # 分类列用众数填充
            for col in categorical_cols:
                if not df[col].mode().empty:
                    df[col] = df[col].fillna(df[col].mode()[0])
                else:
                    df[col] = df[col].fillna("未知")
    
    def prepare_data(self, train_path, test_path):
        """完整的数据预处理流程"""
        print("开始数据预处理...")
        
        # 加载数据
        self.load_data(train_path, test_path)
        
        # 处理训练集和测试集
        for df_name in ['train_data', 'test_data']:
            df = getattr(self, df_name)
            
            print(f"处理{df_name}...")
            
            # 清洗列名
            df = self.clean_column_names(df)
            
            # 删除无用列
            df = df.drop(columns=self.DROP_COLS, errors='ignore')
            
            # 筛选特征（基于缺失率）
            if df_name == 'train_data':
                missing_ratio = df.isnull().mean()
                self.keep_cols = missing_ratio[missing_ratio <= self.MISSING_THRESHOLD].index.tolist()
                essential_cols = [self.target, 'ID', 'text_combined_raw', '室num', '厅num', '厨num', '卫num']
                self.keep_cols = list(set(self.keep_cols + [c for c in essential_cols if c in df.columns]))
                df = df[self.keep_cols]
            else:
                # 测试集对齐训练集特征
                for col in self.keep_cols:
                    if col not in df.columns:
                        df[col] = np.nan
                df = df[self.keep_cols]
            
            # 特征工程
            df = self.extract_numeric_from_unit(df)
            df = self.process_construction_year(df)
            df = self.split_ladder_household(df)
            df = self.extract_room_features(df)
            df = self.process_text_features(df)
            df = self.add_advanced_features(df)
            
            setattr(self, df_name, df)
        
        # 填充缺失值
        self.fill_missing_values()
        
        print("数据预处理完成!")
        return self.train_data, self.test_data
    
    def split_data(self, test_size=0.2, random_state=111):
        """划分训练集和验证集"""
        X = self.train_data.drop(columns=[self.target], errors='ignore')
        y = self.train_data[self.target]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train.select_dtypes(include=[np.number]))
        X_val_scaled = self.scaler.transform(X_val.select_dtypes(include=[np.number]))
        
        return X_train_scaled, X_val_scaled, y_train.values, y_val.values

## 2. 模型训练

In [ ]:
class UnifiedModel:
    """统一模型类 - 整合线性模型和非线性模型"""
    
    def __init__(self, model_type='linear', use_log=True):
        self.model_type = model_type
        self.use_log = use_log
        self.model = None
        self.scaler = StandardScaler()
        
        if model_type == 'linear':
            self.model = LinearRegression()
        elif model_type == 'lasso':
            self.model = Lasso(max_iter=10000)
        elif model_type == 'ridge':
            self.model = Ridge(max_iter=10000)
        elif model_type == 'elasticnet':
            self.model = ElasticNet(max_iter=10000, random_state=42)
        elif model_type == 'lgb':
            self.model = LGBMRegressor(
                objective='regression',
                metric='mae',
                boosting_type='gbdt',
                num_leaves=31,
                learning_rate=0.05,
                n_estimators=1000,
                random_state=111,
                verbose=-1
            )
    
    def fit(self, X, y, val_size=0.2, random_state=111):
        """训练模型"""
        if self.use_log:
            y_fit = np.log1p(y)
        else:
            y_fit = y
        
        X_train, X_val, y_train_fit, y_val_fit = train_test_split(
            X, y_fit, test_size=val_size, random_state=random_state
        )
        
        # 标准化数值特征
        if self.model_type != 'lgb':
            X_train = self.scaler.fit_transform(X_train)
            X_val = self.scaler.transform(X_val)
        
        # 保存原始数据用于评估
        self.X_train, self.X_val = X_train, X_val
        self.y_train_orig = y[np.where(np.isin(np.arange(len(y)), X_train.index) if hasattr(X_train, 'index') else 
                                    np.arange(len(X_train)))]
        self.y_val_orig = y[np.where(np.isin(np.arange(len(y)), X_val.index) if hasattr(X_val, 'index') else 
                                  len(X_train) + np.arange(len(X_val)))]
        
        # 训练模型
        if self.model_type == 'lgb':
            self.model.fit(
                X_train, y_train_fit,
                eval_set=[(X_val, y_val_fit)],
                eval_metric='mae',
                callbacks=[lgb.early_stopping(stopping_rounds=50)]
            )
        else:
            self.model.fit(X_train, y_train_fit)
        
        return self.model
    
    def predict(self, X):
        """预测"""
        if self.model_type != 'lgb':
            X = self.scaler.transform(X)
        
        y_pred = self.model.predict(X)
        
        if self.use_log:
            y_pred = np.expm1(y_pred)
        
        return y_pred
    
    def evaluate(self, X=None, y=None):
        """评估模型"""
        if X is None or y is None:
            X, y = self.X_val, self.y_val_orig
        
        y_pred = self.predict(X)
        
        mae = mean_absolute_error(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        r2 = r2_score(y, y_pred)
        
        return {
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        }
    
    def cross_validate(self, X, y, cv=6):
        """交叉验证"""
        from sklearn.model_selection import cross_val_score
        
        if self.use_log:
            y_cv = np.log1p(y)
        else:
            y_cv = y
        
        if self.model_type != 'lgb':
            X_cv = self.scaler.fit_transform(X)
        else:
            X_cv = X
        
        scores = cross_val_score(self.model, X_cv, y_cv, cv=cv, scoring='neg_mean_absolute_error')
        cv_mae = -scores.mean()
        
        if self.use_log:
            # 近似转换回原始尺度
            cv_mae = np.expm1(cv_mae)
        
        return cv_mae

def train_kmeans_model(data_path, n_clusters=20):
    """训练KMeans聚类模型"""
    data = pd.read_csv(data_path, low_memory=False)
    X = data[['lon', 'lat']].dropna()
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X)
    
    # 保存模型
    os.makedirs('./src/model', exist_ok=True)
    joblib.dump(kmeans, './src/model/kmeans_model.pkl')
    
    print(f"KMeans模型训练完成，聚类数: {n_clusters}")
    return kmeans

## 3. 模型评估与预测

In [ ]:
def evaluate_all_models(train_data, target_column='Price', dataset_name="Dataset"):
    """评估所有模型"""
    print(f"========= Evaluating {dataset_name} Models =========")
    
    # 准备数据
    X = train_data.drop(columns=[target_column], errors='ignore')
    y = train_data[target_column]
    
    # 选择数值列
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X = X[numeric_cols]
    
    models = {
        'OLS': UnifiedModel('linear'),
        'LASSO': UnifiedModel('lasso'),
        'Ridge': UnifiedModel('ridge'),
        'ElasticNet': UnifiedModel('elasticnet'),
        'LightGBM': UnifiedModel('lgb', use_log=False)
    }
    
    results = []
    
    for name, model in tqdm(models.items(), desc="Training Models"):
        # 训练模型
        model.fit(X.values, y.values)
        
        # 样本内评估
        train_metrics = model.evaluate(model.X_train, model.y_train_orig)
        
        # 样本外评估
        val_metrics = model.evaluate()
        
        # 交叉验证
        cv_mae = model.cross_validate(X.values, y.values)
        
        results.append({
            'Model': name,
            'In Sample MAE': round(train_metrics['MAE'], 2),
            'Out of Sample MAE': round(val_metrics['MAE'], 2),
            'Cross-validation MAE': round(cv_mae, 2),
            'R² Score': round(val_metrics['R2'], 4)
        })
    
    # 显示结果
    results_df = pd.DataFrame(results)
    print(f"\n{dataset_name}模型评估结果:")
    print(results_df.to_string(index=False))
    
    return results_df, models

def predict_test_set(models, test_data, target_column='Price', output_dir='./predictions'):
    """预测测试集"""
    os.makedirs(output_dir, exist_ok=True)
    
    # 准备测试数据
    X_test = test_data.drop(columns=[target_column], errors='ignore')
    numeric_cols = X_test.select_dtypes(include=[np.number]).columns
    X_test = X_test[numeric_cols]
    
    # 获取ID
    if 'ID' in test_data.columns:
        ids = test_data['ID']
    else:
        ids = pd.Series(range(len(test_data)), name='ID')
    
    # 为每个模型生成预测
    for name, model in models.items():
        predictions = model.predict(X_test.values)
        
        # 创建结果DataFrame
        result_df = pd.DataFrame({
            'ID': ids,
            'Price': predictions
        })
        
        # 保存结果
        output_path = os.path.join(output_dir, f'{name}_predictions.csv')
        result_df.to_csv(output_path, index=False)
        print(f"{name}预测结果已保存至: {output_path}")
    
    return output_dir

## 4. 完整流程执行

In [ ]:
def main_pipeline():
    """主流程"""
    print("=" * 60)
    print("开始房价与租金预测完整流程")
    print("=" * 60)
    
    # 数据路径配置
    PATHS = {
        'price': {
            'train': './data/raw_data/ruc_Class25Q2_train_price.csv',
            'test': './data/raw_data/ruc_Class25Q2_test_price.csv'
        },
        'rent': {
            'train': './data/raw_data/ruc_Class25Q2_train_rent.csv',
            'test': './data/raw_data/ruc_Class25Q2_test_rent.csv'
        }
    }
    
    results = {}
    
    for data_type in ['price', 'rent']:
        print(f"\n{'='*50}")
        print(f"处理{data_type.upper()}数据")
        print(f"{'='*50}")
        
        # 数据预处理
        processor = UnifiedDataProcessor(data_type=data_type)
        train_data, test_data = processor.prepare_data(
            PATHS[data_type]['train'], 
            PATHS[data_type]['test']
        )
        
        # 模型训练与评估
        results_df, models = evaluate_all_models(
            train_data, 
            dataset_name=f"{data_type.upper()}"
        )
        
        # 测试集预测
        output_dir = predict_test_set(
            models, 
            test_data, 
            output_dir=f'./{data_type}_predictions'
        )
        
        results[data_type] = {
            'results': results_df,
            'models': models,
            'output_dir': output_dir
        }
    
    # 汇总结果
    print("\n" + "="*60)
    print("流程完成汇总")
    print("="*60)
    
    for data_type, result in results.items():
        print(f"\n{data_type.upper()}结果:")
        print(f"最佳模型: {result['results'].iloc[result['results']['Out of Sample MAE'].idxmin()]['Model']}")
        print(f"最佳样本外MAE: {result['results']['Out of Sample MAE'].min():.2f}")
        print(f"预测结果保存至: {result['output_dir']}")
    
    return results

# 执行主流程
if __name__ == "__main__":
    # 首先训练KMeans模型（如果需要）
    try:
        train_kmeans_model('./data/raw_data/ruc_Class25Q2_train_price.csv')
    except Exception as e:
        print(f"KMeans训练失败: {e}")
    
    # 执行主流程
    final_results = main_pipeline()

## 5. 结果分析与可视化

In [ ]:
def analyze_results(final_results):
    """分析结果并生成可视化"""
    
    # 创建结果对比
    comparison_data = []
    
    for data_type, result in final_results.items():
        df = result['results']
        for _, row in df.iterrows():
            comparison_data.append({
                'Dataset': data_type.upper(),
                'Model': row['Model'],
                'Out_of_Sample_MAE': row['Out of Sample MAE'],
                'R2_Score': row['R² Score']
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # 可视化
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. MAE对比
    for i, data_type in enumerate(['PRICE', 'RENT']):
        data = comparison_df[comparison_df['Dataset'] == data_type]
        axes[0, i].bar(data['Model'], data['Out_of_Sample_MAE'], color=['blue', 'orange', 'green', 'red', 'purple'])
        axes[0, i].set_title(f'{data_type} - Out of Sample MAE')
        axes[0, i].set_ylabel('MAE')
        axes[0, i].tick_params(axis='x', rotation=45)
    
    # 2. R²对比
    for i, data_type in enumerate(['PRICE', 'RENT']):
        data = comparison_df[comparison_df['Dataset'] == data_type]
        axes[1, i].bar(data['Model'], data['R2_Score'], color=['blue', 'orange', 'green', 'red', 'purple'])
        axes[1, i].set_title(f'{data_type} - R² Score')
        axes[1, i].set_ylabel('R²')
        axes[1, i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 显示最佳模型
    print("\n最佳模型总结:")
    print("-" * 40)
    
    for data_type in ['PRICE', 'RENT']:
        data = comparison_df[comparison_df['Dataset'] == data_type]
        best_mae = data.loc[data['Out_of_Sample_MAE'].idxmin()]
        best_r2 = data.loc[data['R2_Score'].idxmax()]
        
        print(f"\n{data_type}:")
        print(f"  最低MAE模型: {best_mae['Model']} (MAE: {best_mae['Out_of_Sample_MAE']:.2f})")
        print(f"  最高R²模型: {best_r2['Model']} (R²: {best_r2['R2_Score']:.4f})")
    
    return comparison_df

# 分析结果
if 'final_results' in locals():
    comparison_results = analyze_results(final_results)